# BTC Regime Gate — Google Colab Backtest

**Author:** Ganesh Tilekar  
Repo: https://github.com/GT-2005/btc-regime-gate

### What this does
1. Clones the project from GitHub  
2. You upload BTCUSDT data (`.csv` 15m **or** `.xlsx` 5m/15m)  
3. Runs **short_only** on **2025** (current strategy: TS momentum, one position, SL 1 / TP 3 ATR)

Starting capital = **$100,000** · fees + slippage included.

## 1) Install + clone repo

In [ ]:
!pip -q install pandas numpy scipy scikit-learn lightgbm joblib openpyxl matplotlib

import os, sys

REPO = "btc-regime-gate"
if not os.path.isdir(REPO):
    !git clone https://github.com/GT-2005/btc-regime-gate.git
else:
    !cd btc-regime-gate && git pull

sys.path.insert(0, f"/content/{REPO}/python")
os.chdir(f"/content/{REPO}/python")
print("Working dir:", os.getcwd())

## 2) Upload your data

Upload one of:
- `BTCUSDT_15m_All.csv` (Binance kline style, headerless OK)
- or any year Excel like `BTCUSDT_2025_full.xlsx`

If you already uploaded, set `DATA_PATH` manually in the next cell.

In [ ]:
from google.colab import files

print("Upload BTCUSDT .csv or .xlsx …")
uploaded = files.upload()
assert uploaded, "No file uploaded"

DATA_PATH = list(uploaded.keys())[0]
# Colab saves upload in /content/
if not os.path.isfile(DATA_PATH):
    DATA_PATH = os.path.join("/content", list(uploaded.keys())[0])

print("Using:", DATA_PATH)

## 3) Run 2025 short_only backtest

In [ ]:
import backtest_regime_gate as bt

# Strategy settings (current defaults)
bt.TRADE_MODE = "short_only"
bt.USE_VPIN_FILTER = False
bt.USE_ML_FILTER = False

YEAR = 2025  # change if needed

print("Loading", DATA_PATH)
raw = bt.load_ohlcv(DATA_PATH)
print(f"Raw bars: {len(raw):,} | {raw.index[0]} → {raw.index[-1]}")

df15, df1h, df1d = bt.prepare_timeframes(raw)
print(f"15m bars: {len(df15):,}")

feats = bt.prepare_features(df15, df1h, df1d, year=YEAR)
print(f"Year {YEAR} bars after warmup/filters: {len(feats):,}")

eng = bt.run_backtest(feats)
bt.summarize(eng, feats)

s = bt.run_on_dataframe(feats)
print("\nQuick summary:")
print({
    "end_equity": round(s["end"], 2),
    "return_pct": round(s["ret"], 2),
    "trades": s["n"],
    "win_rate": round(s["wr"], 1),
    "max_dd_pct": round(s["dd"], 2),
    "buy_hold_pct": round(s["bh"], 2),
})

## 4) Equity curve + download trades

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

eq = pd.Series(eng.equity_curve, index=feats.index[: len(eng.equity_curve)], name="equity")
eq.plot(figsize=(12, 4), title=f"Equity curve — {YEAR} short_only")
plt.grid(True, alpha=0.3)
plt.ylabel("USDT")
plt.show()

rows = [
    {
        "side": t.side,
        "playbook": t.playbook,
        "entry_time": t.entry_time,
        "entry_price": t.entry_price,
        "exit_time": t.exit_time,
        "exit_price": t.exit_price,
        "qty": t.qty,
        "pnl": t.pnl,
        "reason": t.reason,
    }
    for t in eng.trades
    if t.exit_time is not None
]
trades = pd.DataFrame(rows)
display(trades.head(20))

trades.to_csv("backtest_trades.csv", index=False)
eq.to_csv("backtest_equity.csv", header=True)
print("Saved backtest_trades.csv and backtest_equity.csv")

from google.colab import files as colab_files
colab_files.download("backtest_trades.csv")
colab_files.download("backtest_equity.csv")

## 5) Optional — compare all modes on 2025

In [ ]:
# Uncomment to compare short_only / long_only / auto / both
# bt.USE_VPIN_FILTER = False
# bt.compare_modes(df15[df15.index.year == YEAR], df1h, df1d)

print("Tip: change TRADE_MODE to 'auto' or YEAR to another year and re-run cell 3.")